# Chapter 2 Advanced Lab: The Bell Inequality from Scratch

**Applied Quantum Computing: A Leader's Guide to the Next Computing Revolution**  
Dr. Ernesto Lee | [Back to Book](https://liquid-books.github.io/applied-quantum-computing)

---

## Objective
Prove that quantum entanglement is real — not just a metaphor — using 20 lines of code.

You will build a Bell state, implement CHSH measurement settings, compute the CHSH value **S**, and show that quantum mechanics violates the classical bound of 2. Most readers never recover from this lab.

## The CHSH Test in Plain English
Two physicists (Alice and Bob) each measure one qubit of an entangled pair. Each independently chooses from two measurement angles. If the qubits were just 'classically correlated' (like pre-scripted answers), the correlations can never exceed **S = 2**. Quantum mechanics predicts **S = 2√2 ≈ 2.828**. We will observe this directly.

## Deliverable
- Working notebook with results
- Chart: S value on simulator vs. real hardware
- **400-word explanation:** Why this experiment falsifies every 'it's just correlated' skeptic

## Estimated Time
60–120 minutes

---

In [ ]:
# Run this first — installs required packages (~60 seconds in Colab)
!pip install -q qiskit qiskit-aer matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print('Qiskit loaded successfully.')
print('We will run on the noiseless AerSimulator (no IBM account needed).')

## Part 1: Build the Bell State |Φ⁺⟩

The Bell state is the simplest maximally entangled state: two qubits whose measurement outcomes are perfectly correlated regardless of which basis you measure in.

In [ ]:
def build_bell_state():
    """Build the Bell state |Φ⁺⟩ = (|00⟩ + |11⟩) / √2"""
    qc = QuantumCircuit(2)
    qc.h(0)       # Put qubit 0 in superposition
    qc.cx(0, 1)   # Entangle qubit 1 with qubit 0 (CNOT)
    return qc

bell = build_bell_state()
print('Bell State Circuit:')
print(bell.draw(output='text'))

## Part 2: CHSH Measurement Settings

The CHSH test uses four measurement angle pairs (a, b), (a, b'), (a', b), (a', b'):
- Alice's angles: **a = 0°**, **a' = 90°**
- Bob's angles: **b = 45°**, **b' = 135°**

The CHSH value is: **S = |E(a,b) - E(a,b') + E(a',b) + E(a',b')|**

Classical bound: **|S| ≤ 2**  
Quantum maximum: **|S| ≤ 2√2 ≈ 2.828**

In [ ]:
def chsh_circuit(alice_angle, bob_angle):
    """Build CHSH measurement circuit for given measurement angles."""
    qc = build_bell_state()
    
    # Rotate measurement basis for Alice (qubit 0)
    qc.ry(-2 * alice_angle, 0)
    
    # Rotate measurement basis for Bob (qubit 1)
    qc.ry(-2 * bob_angle, 1)
    
    qc.measure_all()
    return qc

def compute_expectation(counts, n_shots):
    """Compute E(a,b) = P(same) - P(different)."""
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    return (same - diff) / n_shots

# Define CHSH angle pairs (in radians)
a  = 0
a_ = np.pi / 4      # 45 degrees
b  = np.pi / 8      # 22.5 degrees
b_ = 3 * np.pi / 8  # 67.5 degrees

angle_pairs = [
    ('E(a,b)',   a,  b),
    ('E(a,b\')', a,  b_),
    ('E(a\',b)', a_, b),
    ('E(a\',b\')', a_, b_),
]

print('CHSH measurement angle pairs defined.')
print(f'Alice: a = {np.degrees(a):.1f}°, a\' = {np.degrees(a_):.1f}°')
print(f'Bob:   b = {np.degrees(b):.1f}°, b\' = {np.degrees(b_):.1f}°')

## Part 3: Run on Noiseless Simulator (10,000 shots)

In [ ]:
simulator = AerSimulator()
N_SHOTS = 10000

expectations_sim = {}
for label, alice_angle, bob_angle in angle_pairs:
    qc = chsh_circuit(alice_angle, bob_angle)
    job = simulator.run(qc, shots=N_SHOTS)
    counts = job.result().get_counts()
    E = compute_expectation(counts, N_SHOTS)
    expectations_sim[label] = E
    print(f'{label} = {E:.4f}')

# Compute S
S_sim = abs(
    expectations_sim['E(a,b)'] -
    expectations_sim['E(a,b\')'] +
    expectations_sim['E(a\',b)'] +
    expectations_sim['E(a\',b\')']
)

print(f'\n--- CHSH Results (Noiseless Simulator) ---')
print(f'S = {S_sim:.4f}')
print(f'Classical bound: 2.0000')
print(f'Quantum maximum: {2*np.sqrt(2):.4f}')
print(f'Violation: {"YES ✅" if S_sim > 2 else "NO ❌"} (S > 2 means quantum entanglement confirmed)')

## Part 4: Simulate Noise Effect

Real quantum hardware has noise. We simulate it using Qiskit's noise model to observe how noise pulls S back toward the classical bound of 2.

> **Note:** To run on *actual* IBM quantum hardware, you need a free IBM Quantum account at quantum.ibm.com. The noisy simulation below demonstrates the same effect.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

def run_with_noise(error_rate):
    """Run CHSH with a given depolarizing error rate."""
    noise_model = NoiseModel()
    error_1q = depolarizing_error(error_rate, 1)
    error_2q = depolarizing_error(error_rate * 10, 2)  # 2Q gates are noisier
    noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'ry'])
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    
    noisy_sim = AerSimulator(noise_model=noise_model)
    expectations = {}
    
    for label, alice_angle, bob_angle in angle_pairs:
        qc = chsh_circuit(alice_angle, bob_angle)
        job = noisy_sim.run(qc, shots=N_SHOTS)
        counts = job.result().get_counts()
        expectations[label] = compute_expectation(counts, N_SHOTS)
    
    S = abs(
        expectations['E(a,b)'] - expectations['E(a,b\')'] +
        expectations['E(a\',b)'] + expectations['E(a\',b\')'] 
    )
    return S

# Test various noise levels (0% = perfect, 5% = very noisy)
error_rates = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]
S_values = []

print('Running CHSH at various noise levels...')
for rate in error_rates:
    S = run_with_noise(rate)
    S_values.append(S)
    print(f'  Error rate {rate*100:.1f}%: S = {S:.4f}')

In [ ]:
# Plot: S value vs noise level
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot([r*100 for r in error_rates], S_values, 'o-', color='navy',
        linewidth=2, markersize=8, label='Measured S value')
ax.axhline(y=2*np.sqrt(2), color='green', linestyle='--',
           label=f'Quantum maximum: 2√2 ≈ {2*np.sqrt(2):.3f}')
ax.axhline(y=2.0, color='red', linestyle='--',
           label='Classical bound: S = 2')

ax.fill_between([r*100 for r in error_rates], 2.0, S_values,
                alpha=0.1, color='green', label='Quantum advantage region')

ax.set_xlabel('Gate Error Rate (%)', fontsize=12)
ax.set_ylabel('CHSH Value S', fontsize=12)
ax.set_title('CHSH Bell Inequality: Quantum vs Classical Bound\nHow Noise Degrades Quantum Advantage',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(1.8, 3.0)

plt.tight_layout()
plt.savefig('chsh_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved. Noiseless S = {S_sim:.4f} (quantum bound = {2*np.sqrt(2):.4f})')

## Deliverable: Your 400-Word Explanation

Write your explanation in the cell below. Address:
1. **What did you observe?** What was your S value on the simulator?
2. **What does S > 2 prove?** Why does this rule out 'hidden variables' or pre-scripted correlations?
3. **Why does noise matter commercially?** What does the noise plot tell a CFO about error correction investment?
4. **The skeptic in the meeting:** How would you respond to a board member who says 'it's just pre-correlated data'?

---
**Target:** ~400 words. Write for a non-technical executive audience.

**YOUR EXPLANATION HERE:**

*[Write your 400-word response here...]*